<a href="https://colab.research.google.com/github/marielary/drought-analysis-haute-matsiatra/blob/main/Baseline_traitement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
CE PROGRAMME CALCUL SPI & SPEI — HAUTE MATSIATRA, MADAGASCAR
VERSION FINALE

Indices retenus :
  SPEI-3  de février  → phase critique Jan–Fév  (fenêtre Déc–Jan–Fév)
  SPEI-6  de mai      → bilan saison Déc–Mai
  SPEI-12 de mai      → sécheresse hydrologique de fond

Méthode :
  SPI     → loi Gamma           McKee et al. (1993)
  SPEI    → loi log-logistique  Vicente-Serrano et al. (2010)
  Précip  → CHIRPS v2 (~5 km)
  ET0     → ERA5-Land + Penman-Monteith FAO-56 (altitude SRTM par district)
  Baseline: 1991–2020 (norme OMM 30 ans)
================================================================================
"""

In [1]:
pip install rioxarray -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.6 MB/s eta 0:00:00


In [2]:
!earthengine authenticate

E0000 00:00:1773868324.406934    5041 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773868324.414555    5041 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773868324.435243    5041 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868324.435309    5041 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868324.435314    5041 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868324.435319    5041 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# ============================================================================
# 0. PACKAGES
# ============================================================================

import os, gc, pickle, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

def install_if_missing(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        os.system(f"pip install {pkg} -q")
        print(f"   ✓ {pkg} installé")

for pkg, imp in [
    ('climate-indices', 'climate_indices'),
    ('earthengine-api', 'ee'),
    ('geopandas',       'geopandas'),
    ('scipy',           'scipy'),
    ('matplotlib',      'matplotlib'),
    ('xarray',          'xarray'),
    ('netCDF4',         'netCDF4'),
    ('requests',        'requests'),
]:
    install_if_missing(pkg, imp)

import ee
import xarray as xr
import requests
from climate_indices import compute, indices

print("="*80)
print("CALCUL SPI & SPEI — HAUTE MATSIATRA, MADAGASCAR  [v2]")
print("Indices : SPEI-3 (fév) | SPEI-6 (mai) | SPEI-12 (mai)")
print("="*80)
print(f"Démarrage : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ============================================================================
# 1. CONFIGURATION
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

SHAPEFILE_DISTRICTS = "/content/region_district_HM.shp"
OUTPUT_DIR = "/content/drive/MyDrive/DATA_THESES_MADATLAS/BASELINE/SPI_SPEI_FINAL"
for sub in ['', '/indices', '/validation', '/figures']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

COL_DISTRICT    = "ADM2_EN"
UTM_PROJECTION  = "EPSG:32738"
ALTITUDE_DEFAUT = 1200
ANNEE_DEBUT     = 1991
ANNEE_FIN       = 2020

# Années de sécheresse documentées à Madagascar
ANNEES_SECHES_CONNUES = [1991, 1992, 2001, 2009, 2016, 2019, 2020, 2021]

# Indices cibles — échelle + mois d'extraction
INDICES_CIBLES = [
    {'scale': 3,  'mois': 2, 'label': 'SPEI3_fev',
     'desc': 'Phase critique Jan-Fév (fenêtre Déc-Jan-Fév)'},
    {'scale': 6,  'mois': 5, 'label': 'SPEI6_mai',
     'desc': 'Bilan saison Déc-Mai'},
    {'scale': 12, 'mois': 5, 'label': 'SPEI12_mai',
     'desc': 'Sécheresse hydrologique fond (Juin N-1 → Mai N)'},
]

# Constantes FAO-56
ALBEDO, SIGMA = 0.23, 4.903e-9
P0, T0_K, L_ad = 101.3, 293.15, 0.0065

# Bbox Haute Matsiatra
BBOX = {'lat': (-22.5, -20.0), 'lon': (46.5, 48.0)}

In [ ]:
# ============================================================================
# 2. CHARGEMENT DES DISTRICTS
# ============================================================================

print("\n1. CHARGEMENT DES DISTRICTS")
print("-"*40)

gdf = gpd.read_file(SHAPEFILE_DISTRICTS)
for c in ['ADM2_EN', 'ADM2_FR', 'NAME', 'NOM', 'DISTRICT']:
    if c in gdf.columns:
        COL_DISTRICT = c
        break

if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs("EPSG:4326")

gdf['area_km2'] = gdf.to_crs(UTM_PROJECTION).geometry.area / 1_000_000
print(f"✓ {len(gdf)} districts chargés")

try:
    ee.Initialize()
    print("✓ Earth Engine initialisé")
except:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# ============================================================================
# 3. FONCTIONS SCIENTIFIQUES
# ============================================================================

def get_altitude_district(geom_ee):
    """Altitude moyenne via SRTM 30m — variable par district."""
    try:
        stats = ee.Image("USGS/SRTMGL1_003").reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom_ee,
            scale=500, maxPixels=1e9
        ).getInfo()
        alt = stats.get('elevation')
        return float(alt) if alt is not None else ALTITUDE_DEFAUT
    except:
        return ALTITUDE_DEFAUT

def pression_atmo(z):
    return P0 * ((T0_K - L_ad * z) / T0_K) ** 5.26

def rh_scalaire(T, Td):
    a, b = 17.625, 243.04
    Tc, Tdc = T - 273.15, Td - 273.15
    return 100 * np.exp(a*Tdc/(b+Tdc)) / np.exp(a*Tc/(b+Tc))

def calculer_et0_mensuel(T, Td, u10, v10, ssrd, alt):
    """Penman-Monteith FAO-56 mensuel."""
    try:
        Tc    = T - 273.15
        u2    = np.sqrt(u10**2 + v10**2) * (4.87 / np.log(67.8*10 - 5.42))
        Rs    = ssrd / 1e6
        rh    = np.clip(rh_scalaire(T, Td), 0, 100)
        P     = pression_atmo(alt)
        gamma = 0.000665 * P
        delta = 4098 * (0.6108 * np.exp(17.27*Tc/(Tc+237.3))) / (Tc+237.3)**2
        es    = 0.6108 * np.exp(17.27*Tc/(Tc+237.3))
        ea    = es * rh / 100
        Rns   = (1 - ALBEDO) * Rs
        Rnl   = SIGMA*(Tc+273.16)**4*(0.34-0.14*np.sqrt(ea))*(1.35*Rs/(Rs+0.1)-0.35)
        Rn    = Rns - Rnl
        num   = 0.408*delta*Rn + gamma*(900/(Tc+273))*u2*(es-ea)
        den   = delta + gamma*(1+0.34*u2)
        return float(max(0, num/den))
    except:
        return np.nan

In [ ]:
# ============================================================================
# 4. TÉLÉCHARGEMENT SÉRIES MENSUELLES
# ============================================================================

def telecharger_series_mensuelles(geom_ee, district_nom, alt):
    print(f"\n  📥 {district_nom} — altitude {alt:.0f} m")
    dates  = pd.date_range('1991-01-01', '2020-12-31', freq='MS')
    n      = len(dates)
    precip = np.full(n, np.nan)
    et0    = np.full(n, np.nan)

    # CHIRPS
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD") \
               .filterDate('1991-01-01', '2020-12-31') \
               .select('precipitation')
    print(f"    CHIRPS ...", end='', flush=True)
    ok_p = 0
    for i, date in enumerate(dates):
        an, mo = date.year, date.month
        d0 = f"{an}-{mo:02d}-01"
        d1 = f"{an}-{mo+1:02d}-01" if mo < 12 else f"{an+1}-01-01"
        try:
            val = chirps.filterDate(d0, d1).sum() \
                        .reduceRegion(ee.Reducer.mean(), geom_ee,
                                      5000, maxPixels=1e9).getInfo()
            if val and val.get('precipitation') is not None:
                precip[i] = val['precipitation']
                ok_p += 1
        except:
            continue
    print(f" ✓ ({ok_p}/{n})")

    # ERA5-Land (MONTHLY_AGGR remplace MONTHLY — corrige le DeprecationWarning)
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
             .filterDate('1991-01-01', '2020-12-31') \
             .select(['temperature_2m', 'dewpoint_temperature_2m',
                      'u_component_of_wind_10m', 'v_component_of_wind_10m',
                      'surface_solar_radiation_downwards_sum'])
    print(f"    ERA5-Land ...", end='', flush=True)
    ok_e = 0
    for i, date in enumerate(dates):
        an, mo = date.year, date.month
        d0 = f"{an}-{mo:02d}-01"
        d1 = f"{an}-{mo+1:02d}-01" if mo < 12 else f"{an+1}-01-01"
        try:
            val = era5.filterDate(d0, d1).mean() \
                      .reduceRegion(ee.Reducer.mean(), geom_ee,
                                    10000, maxPixels=1e9).getInfo()
            if val and all(v is not None for v in val.values()):
                et0[i] = calculer_et0_mensuel(
                    val['temperature_2m'],
                    val['dewpoint_temperature_2m'],
                    val['u_component_of_wind_10m'],
                    val['v_component_of_wind_10m'],
                    val['surface_solar_radiation_downwards_sum'],
                    alt
                )
                ok_e += 1
        except:
            continue
    print(f"    ERA5   ✓ ({ok_e}/{n})")

    return pd.DataFrame({
        'date'   : dates,
        'precip' : precip,
        'et0'    : et0,
    })

In [ ]:
# ============================================================================
# 5. CALCUL SPI & SPEI — API climate-indices v2 CORRIGÉE
# ============================================================================

def calculer_spi_spei(df_mensuel):
    """
    Calcule SPI et SPEI aux échelles 3, 6, 12.

    CORRECTION v2 :
      spei() prend désormais deux arguments séparés :
        precips_mm = série de précipitations
        pet_mm     = série d'ET0
      (l'ancienne approche values=P-ET0 ne fonctionne plus en v2.x)

    SPI  → loi Gamma           (McKee et al. 1993)
    SPEI → loi log-logistique  (Vicente-Serrano et al. 2010)
    """
    precip = df_mensuel['precip'].values.astype(float)
    et0    = df_mensuel['et0'].values.astype(float)

    # Remplacer les NaN par interpolation linéaire pour éviter les échecs
    def interpoler(arr):
        s = pd.Series(arr)
        return s.interpolate(method='linear', limit_direction='both').values

    precip_clean = interpoler(precip)
    et0_clean    = interpoler(et0)

    resultats = {'date': df_mensuel['date'].values}

    for scale in [3, 6, 12]:

        # ── SPI ──────────────────────────────────────────────────────
        try:
            resultats[f'SPI_{scale}'] = indices.spi(
                values                   = precip_clean,
                scale                    = scale,
                distribution             = indices.Distribution.gamma,
                data_start_year          = ANNEE_DEBUT,
                calibration_year_initial = ANNEE_DEBUT,
                calibration_year_final   = ANNEE_FIN,
                periodicity              = compute.Periodicity.monthly
            )
            print(f"    ✓ SPI-{scale}")
        except Exception as e:
            print(f"    ⚠️  SPI-{scale} échoué : {e}")
            resultats[f'SPI_{scale}'] = np.full(len(precip), np.nan)

        # ── SPEI — API v2 : precips_mm + pet_mm séparés ──────────────
        try:
            resultats[f'SPEI_{scale}'] = indices.spei(
                precips_mm               = precip_clean,   # ← précipitations
                pet_mm                   = et0_clean,      # ← ET0 (pas P-ET0 !)
                scale                    = scale,
                distribution             = indices.Distribution.pearson,
                periodicity              = compute.Periodicity.monthly,
                data_start_year          = ANNEE_DEBUT,
                calibration_year_initial = ANNEE_DEBUT,
                calibration_year_final   = ANNEE_FIN,
            )
            print(f"    ✓ SPEI-{scale}")
        except Exception as e:
            print(f"    ⚠️  SPEI-{scale} échoué : {e}")
            resultats[f'SPEI_{scale}'] = np.full(len(precip), np.nan)

    return pd.DataFrame(resultats)

In [ ]:
# ============================================================================
# 6. EXTRACTION DES INDICES PAR CAMPAGNE AGRICOLE
# ============================================================================

def extraire_indices_saison(df_indices):
    """
    Extrait les valeurs SPEI d'intérêt par campagne agricole.

    Campagne N = saison débutant en novembre de l'année N.

    ┌──────────────┬───────────────┬──────────────────────────────┐
    │ Indice       │ Extrait en    │ Fenêtre glissante            │
    ├──────────────┼───────────────┼──────────────────────────────┤
    │ SPEI-3 fév   │ Fév N+1       │ Déc N → Jan N+1 → Fév N+1   │
    │ SPEI-6 mai   │ Mai N+1       │ Déc N → … → Mai N+1          │
    │ SPEI-12 mai  │ Mai N+1       │ Juin N → … → Mai N+1         │
    └──────────────┴───────────────┴──────────────────────────────┘
    """
    df = df_indices.copy()
    df['year']  = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month

    def get_val(annee, mois, col):
        mask = (df['year'] == annee) & (df['month'] == mois)
        if mask.any() and col in df.columns:
            v = df.loc[mask, col].values[0]
            return float(v) if not np.isnan(v) else np.nan
        return np.nan

    def categorie(v):
        if np.isnan(v):    return 'NA'
        if v >= -1.0:      return 'Normal'
        if v >= -1.5:      return 'Modérée'
        if v >= -2.0:      return 'Sévère'
        return 'Extrême'

    campagnes = []
    for annee in range(ANNEE_DEBUT, ANNEE_FIN):
        row = {
            'annee_campagne': annee,
            'label_campagne': f"{annee}/{str(annee+1)[-2:]}",
        }
        for cfg in INDICES_CIBLES:
            v = get_val(annee + 1, cfg['mois'], f"SPEI_{cfg['scale']}")
            row[cfg['label']]            = v
            row[f"{cfg['label']}_cat"]   = categorie(v)
        campagnes.append(row)

    return pd.DataFrame(campagnes)

In [ ]:
# ============================================================================
# 7. VALIDATION — SPEI DE RÉFÉRENCE via ERA5-Land natif (GEE)
# ============================================================================
#
# Pourquoi cette approche ?
#   Le réseau Colab bloque les téléchargements directs depuis CSIC/Zenodo
#   (proxy 403). La solution robuste est de calculer un SPEI de référence
#   directement depuis ERA5-Land à sa résolution native (~0.1°) via GEE.
#
# Ce que cela valide :
#   Notre SPEI principal est calculé depuis CHIRPS (précip) + ERA5-Land (ET0).
#   Le SPEI de référence est calculé depuis ERA5-Land (précip + ET0 natifs).
#   → On teste la sensibilité à la SOURCE DE PRÉCIPITATIONS (CHIRPS vs ERA5).
#   Un ρ Spearman élevé confirme que CHIRPS et ERA5 capturent les mêmes
#   signaux de sécheresse, ce qui valide la cohérence de notre approche.
#
# Note :
#   Si on souhate comparer avec SPEIbase CSIC, téléchargez manuellement
#   les fichiers NetCDF depuis https://digital.csic.es/handle/10261/288226
#   et les placez dans OUTPUT_DIR/spei_ref/ avant de relancer.
# ============================================================================

_cache_ref = {}   # cache : évite de re-télécharger pour chaque district

def calculer_spei_reference_era5(geom_ee, district_nom, alt, scale):
    """
    Calcule un SPEI de référence depuis ERA5-Land précipitations natives
    (tp = total precipitation) + ET0 ERA5-Land FAO-56.

    Source précipitations : ERA5-Land 'total_precipitation_sum' (~11 km)
    vs notre source principale CHIRPS (~5 km)

    Retourne une Series mensuelle du SPEI de référence 1991-2020.
    """
    cache_key = f"{district_nom}_{scale}"
    if cache_key in _cache_ref:
        return _cache_ref[cache_key]

    print(f"    ERA5 natif SPEI-{scale} ...", end='', flush=True)

    dates   = pd.date_range('1991-01-01', '2020-12-31', freq='MS')
    n       = len(dates)
    precip  = np.full(n, np.nan)
    et0_ref = np.full(n, np.nan)

    # ERA5-Land MONTHLY_AGGR — précipitations natives + variables ET0
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
             .filterDate('1991-01-01', '2020-12-31') \
             .select([
                 'total_precipitation_sum',          # ← précip ERA5 native (m/mois)
                 'temperature_2m',
                 'dewpoint_temperature_2m',
                 'u_component_of_wind_10m',
                 'v_component_of_wind_10m',
                 'surface_solar_radiation_downwards_sum'
             ])

    ok = 0
    for i, date in enumerate(dates):
        an, mo = date.year, date.month
        d0 = f"{an}-{mo:02d}-01"
        d1 = f"{an}-{mo+1:02d}-01" if mo < 12 else f"{an+1}-01-01"
        try:
            val = era5.filterDate(d0, d1).mean() \
                      .reduceRegion(ee.Reducer.mean(), geom_ee,
                                    10000, maxPixels=1e9).getInfo()
            if val and all(v is not None for v in val.values()):
                # Convertir m → mm
                precip[i]  = val['total_precipitation_sum'] * 1000.0
                et0_ref[i] = calculer_et0_mensuel(
                    val['temperature_2m'],
                    val['dewpoint_temperature_2m'],
                    val['u_component_of_wind_10m'],
                    val['v_component_of_wind_10m'],
                    val['surface_solar_radiation_downwards_sum'],
                    alt
                )
                ok += 1
        except:
            continue

    print(f" ✓ ({ok}/{n})")

    if ok < n * 0.5:
        print(f"    ⚠️  Trop de données manquantes pour la référence ERA5")
        _cache_ref[cache_key] = None
        return None

    # Interpolation des valeurs manquantes
    def interp(arr):
        return pd.Series(arr).interpolate('linear', limit_direction='both').values

    precip_c = interp(precip)
    et0_c    = interp(et0_ref)

    # Calcul SPEI de référence
    try:
        spei_ref_vals = indices.spei(
            precips_mm               = precip_c,
            pet_mm                   = et0_c,
            scale                    = scale,
            distribution             = indices.Distribution.pearson,
            periodicity              = compute.Periodicity.monthly,
            data_start_year          = ANNEE_DEBUT,
            calibration_year_initial = ANNEE_DEBUT,
            calibration_year_final   = ANNEE_FIN,
        )
        result = pd.Series(
            spei_ref_vals.astype(float),
            index=dates,
            name=f'SPEI_{scale}_ref'
        )
        _cache_ref[cache_key] = result
        return result
    except Exception as e:
        print(f"    ⚠️  Calcul SPEI référence échoué : {e}")
        _cache_ref[cache_key] = None
        return None


def charger_spei_csic_si_disponible(scale):
    """
    Tente de charger le SPEIbase CSIC depuis le Drive si l'utilisateur
    l'a téléchargé manuellement.

    Chemin attendu : OUTPUT_DIR/spei_ref/spei{scale:02d}.nc
    Téléchargement : https://digital.csic.es/handle/10261/288226
    """
    path = f"{OUTPUT_DIR}/spei_ref/spei{scale:02d}.nc"
    if not os.path.exists(path):
        return None
    try:
        ds  = xr.open_dataset(path, engine='netcdf4')
        print(f"    ✓ SPEIbase CSIC trouvé : {path}")
        return ds
    except Exception as e:
        print(f"    ⚠️  Fichier CSIC illisible : {e}")
        return None


def extraire_spei_ref_district(geom_ee, centroide_lat, centroide_lon,
                                district_nom, alt, scale):
    """
    Stratégie à deux niveaux :
      1. SPEIbase CSIC (si fichier NetCDF présent dans Drive) → référence externe
      2. SPEI ERA5-Land natif via GEE                         → référence interne

    La stratégie 2 est toujours disponible et suffisante pour valider
    la cohérence de notre approche CHIRPS vs ERA5.
    """
    # Niveau 1 : CSIC si disponible
    ds_csic = charger_spei_csic_si_disponible(scale)
    if ds_csic is not None:
        try:
            varnames = ['spei', 'SPEI', f'spei{scale:02d}']
            var = next((v for v in varnames if v in ds_csic.data_vars), None)
            if var:
                da = ds_csic[var].sel(
                    lat=centroide_lat, lon=centroide_lon, method='nearest'
                ).sel(time=slice('1991-01-01', '2020-12-31'))
                vals  = da.values.astype(float)
                dates = pd.to_datetime(da.time.values)
                print(f"    ✓ CSIC SPEI-{scale} ({len(vals)} mois) [source externe]")
                return pd.Series(vals, index=dates, name=f'SPEI_{scale}_ref'), 'CSIC'
        except Exception as e:
            print(f"    ⚠️  Extraction CSIC échouée ({e}) → ERA5 natif")

    # Niveau 2 : ERA5-Land natif (toujours disponible via GEE)
    ref = calculer_spei_reference_era5(geom_ee, district_nom, alt, scale)
    return ref, 'ERA5-natif'

def valider_spei(df_indices, spei_ref, district_nom, scale, source='ERA5-natif'):
    """
    Validation croisée SPEI calculé vs SPEI référence CSIC.
    Métriques : ρ Spearman, r Pearson, RMSE, biais, détection années sèches.
    """
    col = f'SPEI_{scale}'
    if spei_ref is None or col not in df_indices.columns:
        return {}

    df_v = df_indices.set_index('date')[[col]] \
                     .join(spei_ref.rename('ref'), how='inner').dropna()

    if len(df_v) < 24:
        print(f"    ⚠️  Insuffisant pour valider ({len(df_v)} mois)")
        return {}

    calc = df_v[col].values
    ref  = df_v['ref'].values

    rho, p_rho = spearmanr(calc, ref)
    r,   _     = pearsonr(calc, ref)
    rmse       = float(np.sqrt(np.mean((calc - ref)**2)))
    biais      = float(np.mean(calc - ref))

    detected, testable = 0, 0
    for an in ANNEES_SECHES_CONNUES:
        mask = df_v.index.year == an
        if mask.any():
            testable += 1
            if df_v.loc[mask, col].min() <= -1.0:
                detected += 1

    detection_pct = detected / testable * 100 if testable else np.nan
    qualite = 'BONNE' if rho >= 0.7 else ('ACCEPTABLE' if rho >= 0.5 else 'FAIBLE')

    print(f"\n    ── Validation {district_nom} SPEI-{scale} ──")
    print(f"       ρ Spearman = {rho:.3f}  (p={p_rho:.4f})  |  Qualité : {qualite}")
    print(f"       r Pearson  = {r:.3f}  |  RMSE = {rmse:.3f}  |  Biais = {biais:+.3f}")
    print(f"       Détection  = {detected}/{testable} ({detection_pct:.0f}%)")

    return {
        'district': district_nom, 'scale': scale, 'n_mois': len(df_v),
        'spearman_rho': round(rho, 3), 'spearman_p': round(p_rho, 4),
        'pearson_r': round(r, 3), 'rmse': round(rmse, 3), 'biais': round(biais, 3),
        'detection': f"{detected}/{testable}",
        'detection_pct': round(detection_pct, 1), 'qualite': qualite, 'source': source
    }

In [ ]:
# ============================================================================
# 8. VISUALISATIONS
# ============================================================================

def tracer_validation(df_indices, spei_ref, district_nom, scale, source, output_dir):
    col = f'SPEI_{scale}'
    if spei_ref is None or col not in df_indices.columns:
        return

    df_v = df_indices.set_index('date')[[col]] \
                     .join(spei_ref.rename('ref'), how='inner').dropna()
    if len(df_v) < 12:
        return

    fig = plt.figure(figsize=(16, 8))
    gs  = gridspec.GridSpec(2, 2)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df_v.index, df_v['ref'], color='gray', lw=1.2, alpha=0.7,
             label=f'SPEI référence ({source})')
    ax1.plot(df_v.index, df_v[col], color='steelblue', lw=1.5,
             label='SPEI calculé (CHIRPS + ERA5-Land)')
    ax1.axhline(-1.0, color='orange', ls='--', lw=1, label='−1 (modéré)')
    ax1.axhline(-1.5, color='red',    ls='--', lw=1, label='−1.5 (sévère)')
    ax1.axhline( 0,   color='k',      ls='-',  lw=0.5)
    for an in ANNEES_SECHES_CONNUES:
        ax1.axvline(pd.Timestamp(f'{an}-01-01'), color='red', alpha=0.12, lw=1)
    ax1.set_title(f'{district_nom} — SPEI-{scale} calculé vs. CSIC (1991–2020)',
                  fontsize=12, fontweight='bold')
    ax1.set_ylabel('SPEI')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.scatter(df_v['ref'], df_v[col], alpha=0.4, s=18, color='steelblue')
    lim = max(abs(df_v.values).max(), 2.5) * 1.05
    ax2.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
    rho, _ = spearmanr(df_v[col], df_v['ref'])
    ax2.set_xlabel(f'SPEI référence ({source})')
    ax2.set_ylabel('SPEI calculé')
    ax2.set_title(f'Scatter — ρ Spearman = {rho:.3f}')
    ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    res = df_v[col] - df_v['ref']
    ax3.hist(res, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax3.axvline(0, color='red', ls='--')
    ax3.set_xlabel('Résidu (calculé − référence)')
    ax3.set_ylabel('Fréquence')
    ax3.set_title(f'Résidus — biais = {res.mean():+.3f}')
    ax3.grid(alpha=0.3)

    plt.tight_layout()
    fname = f"{output_dir}/figures/valid_{district_nom.replace(' ','_')}_SPEI{scale}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    📊 {fname}")

def tracer_indices_saison(df_saison, district_nom, output_dir):
    cols = ['SPEI3_fev', 'SPEI6_mai', 'SPEI12_mai']
    labs = ['SPEI-3 Fév  (Phase critique Déc–Jan–Fév)',
            'SPEI-6 Mai  (Bilan saison Déc–Mai)',
            'SPEI-12 Mai (Fond hydrologique Juin–Mai)']
    annees = df_saison['annee_campagne'].values + 1
    x = np.arange(len(annees))

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    for ax, col, lab in zip(axes, cols, labs):
        vals  = df_saison[col].values
        cmap  = ['#d32f2f' if v <= -1.5 else
                 '#ef9a9a' if v <= -1.0 else
                 '#1976d2' if v >= 1.0  else
                 '#90a4ae' for v in np.nan_to_num(vals)]
        ax.bar(x, vals, color=cmap, width=0.7, edgecolor='white', linewidth=0.3)
        ax.axhline(-1.0, color='orange', ls='--', lw=1,   label='−1 (modéré)')
        ax.axhline(-1.5, color='red',    ls='--', lw=0.8, label='−1.5 (sévère)')
        ax.axhline( 0,   color='k',      ls='-',  lw=0.5)
        ax.set_ylabel('SPEI')
        ax.set_ylim(-3.5, 3.5)
        ax.set_title(lab, fontsize=10)
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(axis='y', alpha=0.3)

    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels([str(a) for a in annees],
                              rotation=45, ha='right', fontsize=8)
    axes[-1].set_xlabel('Année (fin de campagne agricole)')
    axes[0].set_title(f'{district_nom} — Indices SPEI par campagne (1991–2020)',
                      fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f"{output_dir}/figures/saison_{district_nom.replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    📊 {fname}")

In [ ]:
# ============================================================================
# 9. BOUCLE PRINCIPALE
# ============================================================================

print("\n2. TRAITEMENT PAR DISTRICT")
print("="*60)

all_validation = []
all_indices    = {}
all_saisons    = {}

for idx, row in gdf.iterrows():
    district = row[COL_DISTRICT]
    print(f"\n{'='*60}")
    print(f"DISTRICT : {district}")
    print(f"{'='*60}")

    coords  = list(row.geometry.exterior.coords)
    geom_ee = ee.Geometry.Polygon(coords)

    # Altitude variable via SRTM
    alt = get_altitude_district(geom_ee)
    print(f"  Altitude SRTM : {alt:.0f} m")

    # Centroïde pour extraction SPEI CSIC
    centroid     = row.geometry.centroid
    cent_lat     = centroid.y
    cent_lon     = centroid.x

    # Séries brutes
    df_mensuel = telecharger_series_mensuelles(geom_ee, district, alt)

    if df_mensuel['precip'].isna().mean() > 0.5:
        print(f"  ⚠️  >50% données manquantes — district ignoré")
        continue

    df_mensuel.to_csv(
        f"{OUTPUT_DIR}/indices/{district.replace(' ','_')}_brut.csv",
        index=False)

    # SPI & SPEI
    print(f"\n  Calcul SPI & SPEI...")
    df_idx = calculer_spi_spei(df_mensuel)

    # Indices saisonniers
    print(f"  Extraction indices saisonniers...")
    df_saison = extraire_indices_saison(df_idx)

    # Sauvegardes
    df_mensuel.merge(df_idx, on='date', how='left').to_csv(
        f"{OUTPUT_DIR}/indices/{district.replace(' ','_')}_indices_mensuels.csv",
        index=False)
    df_saison.to_csv(
        f"{OUTPUT_DIR}/indices/{district.replace(' ','_')}_indices_saison.csv",
        index=False)

    all_indices[district] = df_idx
    all_saisons[district] = df_saison

    # Résumé console
    print(f"\n  {'Campagne':<12} {'SPEI3_fev':>10} {'SPEI6_mai':>10} "
          f"{'SPEI12_mai':>11}  {'Cat. fév':>10}")
    print(f"  {'-'*58}")
    for _, r2 in df_saison.iterrows():
        print(f"  {r2['label_campagne']:<12} "
              f"{r2['SPEI3_fev']:>10.2f} "
              f"{r2['SPEI6_mai']:>10.2f} "
              f"{r2['SPEI12_mai']:>11.2f}  "
              f"{r2['SPEI3_fev_cat']:>10}")

    tracer_indices_saison(df_saison, district, OUTPUT_DIR)

    # Validation croisée
    print(f"\n  Validation croisée SPEI (ERA5-natif ou CSIC si disponible)...")
    for cfg in INDICES_CIBLES:
        scale    = cfg['scale']
        spei_ref, source = extraire_spei_ref_district(
            geom_ee, cent_lat, cent_lon, district, alt, scale)
        val_res  = valider_spei(df_idx, spei_ref, district, scale, source)
        if val_res:
            all_validation.append(val_res)
            tracer_validation(df_idx, spei_ref, district, scale, source, OUTPUT_DIR)

    gc.collect()

In [ ]:
# ============================================================================
# 10. RAPPORT CONSOLIDÉ
# ============================================================================

print("\n3. RAPPORT DE VALIDATION")
print("="*60)

df_val = pd.DataFrame(all_validation)
if not df_val.empty:
    df_val.to_csv(f"{OUTPUT_DIR}/validation/rapport_validation.csv", index=False)
    print(df_val[['district','scale','source','spearman_rho','rmse','detection_pct','qualite']]
          .to_string(index=False))
    faibles = df_val[df_val['qualite'] == 'FAIBLE']
    if not faibles.empty:
        print(f"\n  ⚠️  Qualité FAIBLE pour :")
        print(faibles[['district','scale','spearman_rho']].to_string(index=False))

In [ ]:

# ============================================================================
# 11. SAUVEGARDE FINALE
# ============================================================================

print("\n4. SAUVEGARDE FINALE")

with open(f"{OUTPUT_DIR}/all_indices_mensuels.pkl", 'wb') as f:
    pickle.dump(all_indices, f)
with open(f"{OUTPUT_DIR}/all_indices_saison.pkl", 'wb') as f:
    pickle.dump(all_saisons, f)

if all_saisons:
    pd.concat(
        [df.assign(district=d) for d, df in all_saisons.items()],
        ignore_index=True
    ).to_csv(f"{OUTPUT_DIR}/indices_saison_tous_districts.csv", index=False)

with open(f"{OUTPUT_DIR}/metadata.txt", 'w') as f:
    f.write("SPI & SPEI — HAUTE MATSIATRA (v2 CORRIGÉE)\n")
    f.write("="*50 + "\n")
    f.write(f"Date              : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Baseline          : {ANNEE_DEBUT}–{ANNEE_FIN} (norme OMM)\n")
    f.write(f"SPI               : loi Gamma — McKee et al. (1993)\n")
    f.write(f"SPEI              : loi log-logistique — Vicente-Serrano et al. (2010)\n")
    f.write(f"API               : climate-indices v2 (precips_mm + pet_mm séparés)\n")
    f.write(f"Précipitations    : CHIRPS v2 (~5 km)\n")
    f.write(f"ET0               : ERA5-Land MONTHLY_AGGR + Penman-Monteith FAO-56\n")
    f.write(f"Altitude          : SRTM 30m via GEE (variable par district)\n")
    f.write(f"Validation        : SPEIbase v2.9 CSIC (NetCDF direct)\n\n")
    f.write("INDICES RETENUS :\n")
    for cfg in INDICES_CIBLES:
        f.write(f"  {cfg['label']:<14} SPEI-{cfg['scale']} mois={cfg['mois']:02d} "
                f"→ {cfg['desc']}\n")

print(f"\n{'='*80}")
print(f"✅ TERMINÉ")
print(f"📁 {OUTPUT_DIR}")
print(f"   ├── indices/[district]_brut.csv")
print(f"   ├── indices/[district]_indices_mensuels.csv")
print(f"   ├── indices/[district]_indices_saison.csv")
print(f"   ├── indices_saison_tous_districts.csv")
print(f"   ├── validation/rapport_validation.csv")
print(f"   ├── figures/")
print(f"   └── metadata.txt")
print(f"⏱️  Fin : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}")

CALCUL SPI & SPEI — HAUTE MATSIATRA, MADAGASCAR  [v2]
Indices : SPEI-3 (fév) | SPEI-6 (mai) | SPEI-12 (mai)
Démarrage : 2026-03-18 21:38:25
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. CHARGEMENT DES DISTRICTS
----------------------------------------
✓ 7 districts chargés
✓ Earth Engine initialisé

2. TRAITEMENT PAR DISTRICT

DISTRICT : Fianarantsoa I
  Altitude SRTM : 1153 m

  📥 Fianarantsoa I — altitude 1153 m
    CHIRPS ... ✓ (360/360)
    ERA5-Land ...    ERA5   ✓ (360/360)

  Calcul SPI & SPEI...
    ✓ SPI-3
    ✓ SPEI-3
    ✓ SPI-6
    ✓ SPEI-6
    ✓ SPI-12
    ✓ SPEI-12
  Extraction indices saisonniers...

  Campagne      SPEI3_fev  SPEI6_mai  SPEI12_mai    Cat. fév
  ----------------------------------------------------------
  1991/92           -1.53      -1.59       -2.03      Sévère
  1992/93           -0.66      -0.87       -0.62      Normal
  1993/94            0.81       0.77        0.